# Gold Dimension — Medication Route Semantic Reference

**Target table:** `cdr_mimiciv.gold.dim_medication_route`

This notebook creates a Gold reference/dimension table for medication administration routes.
It is designed to make raw route codes easier to interpret in AI/BI dashboards and Databricks Genie.

The source vocabulary is embedded directly in this notebook for reproducibility.

The table preserves:
- raw route code
- human-readable route name
- functional route group
- UMLS CUI
- synonyms

It also adds a normalized route code and a flattened synonym string for easier joins and semantic search.

In [0]:
import json
from pyspark.sql import functions as F
from pyspark.sql import types as T

TARGET_TABLE = "cdr_mimiciv.gold.dim_medication_route"

# Embedded source vocabulary.
ROUTE_VOCABULARY_JSON = r"""
[
  {
    "@type": "RouteVocabulary",
    "rawCode": "PO",
    "fullForm": "Per Os (By mouth / Oral Delivery)",
    "functionalGroup": "Enteral / Oral",
    "umlsCUI": "C38288",
    "synonyms": [
      "by mouth",
      "oral",
      "per os",
      "swallow",
      "mouth",
      "po"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "ORAL",
    "fullForm": "Oral Administration",
    "functionalGroup": "Enteral / Oral",
    "umlsCUI": "C38288",
    "synonyms": [
      "oral",
      "mouth",
      "by mouth",
      "swallowed"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "SL",
    "fullForm": "Sublingual (Dissolved under the tongue)",
    "functionalGroup": "Enteral / Oral",
    "umlsCUI": "C38300",
    "synonyms": [
      "sublingual",
      "under the tongue",
      "sub-lingual"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "BUCCAL",
    "fullForm": "Buccal (Dissolved between cheek and gums)",
    "functionalGroup": "Enteral / Oral",
    "umlsCUI": "C38210",
    "synonyms": [
      "buccal",
      "cheek",
      "inside cheek",
      "bu"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "BU",
    "fullForm": "Buccal Delivery Variant",
    "functionalGroup": "Enteral / Oral",
    "umlsCUI": "C38210",
    "synonyms": [
      "bu",
      "buccal",
      "cheek"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "NG",
    "fullForm": "Nasogastric Tube",
    "functionalGroup": "Tube / Enteral",
    "umlsCUI": "C38274",
    "synonyms": [
      "nasogastric",
      "ng tube",
      "feeding tube",
      "stomach tube"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "OG",
    "fullForm": "Orogastric Tube",
    "functionalGroup": "Tube / Enteral",
    "umlsCUI": "C1167440",
    "synonyms": [
      "orogastric",
      "og tube",
      "oral feeding tube"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "ND",
    "fullForm": "Nasoduodenal Tube",
    "functionalGroup": "Tube / Enteral",
    "umlsCUI": "C1167439",
    "synonyms": [
      "nasoduodenal",
      "nd tube",
      "duodenal tube"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "J TUBE",
    "fullForm": "Jejunostomy Tube",
    "functionalGroup": "Tube / Enteral",
    "umlsCUI": "C38258",
    "synonyms": [
      "jejunostomy",
      "j-tube",
      "j tube",
      "jejunum line"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "G TUBE",
    "fullForm": "Gastrostomy Tube",
    "functionalGroup": "Tube / Enteral",
    "umlsCUI": "C38243",
    "synonyms": [
      "gastrostomy",
      "g-tube",
      "g tube",
      "peg tube"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "PO/NG",
    "fullForm": "By mouth or Nasogastric Tube",
    "functionalGroup": "Tube / Enteral",
    "umlsCUI": "C5232148",
    "synonyms": [
      "po or ng",
      "oral or nasogastric",
      "mouth or feeding tube"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "PO/OG",
    "fullForm": "By mouth or Orogastric Tube",
    "functionalGroup": "Tube / Enteral",
    "umlsCUI": "C5232148",
    "synonyms": [
      "po or og",
      "oral or orogastric"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "NG/OG",
    "fullForm": "Nasogastric or Orogastric Tube",
    "functionalGroup": "Tube / Enteral",
    "umlsCUI": "C5232148",
    "synonyms": [
      "ng or og",
      "nasogastric or orogastric",
      "gastric tube"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "PO OR ENTERAL TUBE",
    "fullForm": "By mouth or Enteral Feeding Tube",
    "functionalGroup": "Tube / Enteral",
    "umlsCUI": "C5232148",
    "synonyms": [
      "oral or enteral",
      "mouth or tube",
      "po or feeding tube"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "ENTERAL TUBE ONLY – NOT ORAL",
    "fullForm": "Enteral Tube Only (Strictly not oral)",
    "functionalGroup": "Tube / Enteral",
    "umlsCUI": "C5232148",
    "synonyms": [
      "tube only",
      "not po",
      "enteral tube mapping",
      "strict tube"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "IV",
    "fullForm": "Intravenous",
    "functionalGroup": "Intravenous (IV)",
    "umlsCUI": "C38112",
    "synonyms": [
      "intravenous",
      "iv",
      "injected in vein",
      "intravenous line"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "IV INFUSION",
    "fullForm": "Intravenous Infusion",
    "functionalGroup": "Intravenous (IV)",
    "umlsCUI": "C38250",
    "synonyms": [
      "iv continuous",
      "iv infusion",
      "drip",
      "intravenous drip"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "IV BOLUS",
    "fullForm": "Intravenous Bolus",
    "functionalGroup": "Intravenous (IV)",
    "umlsCUI": "C1512948",
    "synonyms": [
      "iv bolus",
      "iv push",
      "rapid iv injection",
      "ivb"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "IV DRIP",
    "fullForm": "Intravenous Drip",
    "functionalGroup": "Intravenous (IV)",
    "umlsCUI": "C38250",
    "synonyms": [
      "iv drip",
      "gravity infusion",
      "microdrip"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "IVCI",
    "fullForm": "Intravenous Continuous Infusion",
    "functionalGroup": "Intravenous (IV)",
    "umlsCUI": "C38250",
    "synonyms": [
      "ivci",
      "continuous iv infusion",
      "constant drip"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "PB",
    "fullForm": "Piggyback (Secondary IV line)",
    "functionalGroup": "Intravenous (IV)",
    "umlsCUI": "C0181912",
    "synonyms": [
      "piggyback",
      "ivpb",
      "secondary line",
      "secondary infusion"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "IVPCA",
    "fullForm": "Intravenous Patient-Controlled Analgesia",
    "functionalGroup": "Intravenous (IV)",
    "umlsCUI": "C0182414",
    "synonyms": [
      "ivpca",
      "patient controlled analgesia",
      "pain pump",
      "iv pca"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "IVS",
    "fullForm": "Intravenous Side-port",
    "functionalGroup": "Intravenous (IV)",
    "umlsCUI": "C1167341",
    "synonyms": [
      "ivs",
      "side port",
      "line connector port"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "SIDE PORT",
    "fullForm": "Intravenous Side-port Delivery",
    "functionalGroup": "Intravenous (IV)",
    "umlsCUI": "C1167341",
    "synonyms": [
      "side port",
      "line adapter",
      "secondary entry port"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "FLUSH",
    "fullForm": "Intravenous Line Flush",
    "functionalGroup": "Intravenous (IV)",
    "umlsCUI": "C0559740",
    "synonyms": [
      "flush",
      "line flush",
      "saline flush",
      "heparin lock flush",
      "iv flush"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "IV FLUSH",
    "fullForm": "Intravenous Line Flush Target",
    "functionalGroup": "Intravenous (IV)",
    "umlsCUI": "C0559740",
    "synonyms": [
      "iv flush",
      "catheter flush",
      "line clearing"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "IV FLUSH*",
    "fullForm": "Intravenous Line Flush Protocol Entry",
    "functionalGroup": "Intravenous (IV)",
    "umlsCUI": "C0559740",
    "synonyms": [
      "iv flush*",
      "flush variable",
      "asterisk flush"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "PO/IV",
    "fullForm": "By mouth or Intravenous",
    "functionalGroup": "Mixed Delivery",
    "umlsCUI": "C5232148",
    "synonyms": [
      "po or iv",
      "oral or intravenous",
      "switch route"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "DLPICC",
    "fullForm": "Dual-Lumen Peripherally Inserted Central Catheter",
    "functionalGroup": "Central/Vascular Line",
    "umlsCUI": "C1167323",
    "synonyms": [
      "dlpicc",
      "picc line",
      "peripherally inserted central catheter",
      "dual lumen picc"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "IJ",
    "fullForm": "Internal Jugular Line",
    "functionalGroup": "Central/Vascular Line",
    "umlsCUI": "C1167175",
    "synonyms": [
      "ij line",
      "internal jugular",
      "jugular central venous line",
      "ij cvc"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "FEMORAL VEIN",
    "fullForm": "Femoral Vein Central Line",
    "functionalGroup": "Central/Vascular Line",
    "umlsCUI": "C1167174",
    "synonyms": [
      "femoral line",
      "femoral vein catheter",
      "femoral cvc"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "ARTERIAL LINE",
    "fullForm": "Arterial Line Infusion",
    "functionalGroup": "Central/Vascular Line",
    "umlsCUI": "C0180479",
    "synonyms": [
      "art line",
      "a-line",
      "arterial catheter",
      "intra-arterial infusion"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "IA",
    "fullForm": "Intra-arterial (Into an artery)",
    "functionalGroup": "Central/Vascular Line",
    "umlsCUI": "C38247",
    "synonyms": [
      "intra-arterial",
      "ia",
      "into artery"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "INTRA-ARTERIALLY",
    "fullForm": "Intra-arterial Administration Text",
    "functionalGroup": "Central/Vascular Line",
    "umlsCUI": "C38247",
    "synonyms": [
      "intra-arterially",
      "arterial injection"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "IM",
    "fullForm": "Intramuscular (Injection into muscle)",
    "functionalGroup": "Injectable / Catheter",
    "umlsCUI": "C38248",
    "synonyms": [
      "intramuscular",
      "im",
      "muscle injection",
      "intramuscularly"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "SUBCUT",
    "fullForm": "Subcutaneous (Under the skin)",
    "functionalGroup": "Injectable / Catheter",
    "umlsCUI": "C38299",
    "synonyms": [
      "subcut",
      "subcutaneous",
      "sq",
      "sub-q",
      "sc"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "SC",
    "fullForm": "Subcutaneous Standard Abbreviation",
    "functionalGroup": "Injectable / Catheter",
    "umlsCUI": "C38299",
    "synonyms": [
      "sc",
      "subcutaneous",
      "subq"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "SUBDERMAL",
    "fullForm": "Subdermal (Deep layer of skin)",
    "functionalGroup": "Injectable / Catheter",
    "umlsCUI": "C1167191",
    "synonyms": [
      "subdermal",
      "under the dermis",
      "skin layer"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "ID",
    "fullForm": "Intradermal (Into the skin layer)",
    "functionalGroup": "Injectable / Catheter",
    "umlsCUI": "C38249",
    "synonyms": [
      "intradermal",
      "id",
      "skin injection",
      "tb test route"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "SC INFUSION",
    "fullForm": "Subcutaneous Infusion",
    "functionalGroup": "Injectable / Catheter",
    "umlsCUI": "C1167385",
    "synonyms": [
      "subcut infusion",
      "continuous subcut",
      "sc infusion"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "SUBCUT INFUSION",
    "fullForm": "Subcutaneous Infusion Full Text",
    "functionalGroup": "Injectable / Catheter",
    "umlsCUI": "C1167385",
    "synonyms": [
      "subcutaneous infusion",
      "subq continuous"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "SCPCA",
    "fullForm": "Subcutaneous Patient-Controlled Analgesia",
    "functionalGroup": "Injectable / Catheter",
    "umlsCUI": "C1167332",
    "synonyms": [
      "scpca",
      "subcutaneous pca",
      "subq pain pump"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "SCPUMP",
    "fullForm": "Subcutaneous Pump",
    "functionalGroup": "Injectable / Catheter",
    "umlsCUI": "C1167385",
    "synonyms": [
      "scpump",
      "subq pump",
      "insulin pump route"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "ED",
    "fullForm": "Epidural",
    "functionalGroup": "Regional Block / Spinal",
    "umlsCUI": "C38226",
    "synonyms": [
      "epidural",
      "ed",
      "epidural space",
      "spinal block"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "EDPCA",
    "fullForm": "Epidural Patient-Controlled Analgesia",
    "functionalGroup": "Regional Block / Spinal",
    "umlsCUI": "C1167325",
    "synonyms": [
      "edpca",
      "epidural pca",
      "labor epidural pump"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "IT",
    "fullForm": "Intrathecal (Into the spinal canal)",
    "functionalGroup": "Regional Block / Spinal",
    "umlsCUI": "C38252",
    "synonyms": [
      "intrathecal",
      "it",
      "spinal fluid injection",
      "csf injection"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "ITPCA",
    "fullForm": "Intrathecal Patient-Controlled Analgesia",
    "functionalGroup": "Regional Block / Spinal",
    "umlsCUI": "C1167329",
    "synonyms": [
      "itpca",
      "intrathecal pca",
      "spinal pca"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "IT IMPLANTED",
    "fullForm": "Intrathecal Implanted Pump",
    "functionalGroup": "Regional Block / Spinal",
    "umlsCUI": "C1167329",
    "synonyms": [
      "it implanted",
      "implanted intrathecal pump",
      "baclofen pump route"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "INTERSPACE",
    "fullForm": "Interspace (Epidural or spinal space landmarks)",
    "functionalGroup": "Regional Block / Spinal",
    "umlsCUI": "C1167173",
    "synonyms": [
      "interspace",
      "inter-spinal space",
      "spinal vertebrae injection"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "LUMBAR PLEXUS",
    "fullForm": "Lumbar Plexus Nerve Block",
    "functionalGroup": "Regional Block / Spinal",
    "umlsCUI": "C1167177",
    "synonyms": [
      "lumbar plexus",
      "psoas compartment block",
      "lumbar nerve infusion"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "PERIPHNERVE",
    "fullForm": "Peripheral Nerve Block",
    "functionalGroup": "Regional Block / Spinal",
    "umlsCUI": "C1167181",
    "synonyms": [
      "periphnerve",
      "peripheral nerve block",
      "nerve catheter block",
      "femoral nerve block"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "IC",
    "fullForm": "Intracardiac (Directly into the heart)",
    "functionalGroup": "Specialized Organs / Cavities",
    "umlsCUI": "C38218",
    "synonyms": [
      "intracardiac",
      "ic",
      "heart injection",
      "direct cardiac"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "IP",
    "fullForm": "Intraperitoneal (Into the peritoneal cavity)",
    "functionalGroup": "Specialized Organs / Cavities",
    "umlsCUI": "C38251",
    "synonyms": [
      "intraperitoneal",
      "ip",
      "peritoneal cavity",
      "abdominal infusion"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "PL",
    "fullForm": "Intrapleural (Into the chest/pleural cavity)",
    "functionalGroup": "Specialized Organs / Cavities",
    "umlsCUI": "C38254",
    "synonyms": [
      "intrapleural",
      "pl",
      "pleural cavity",
      "chest tube instillation"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "INTRAPLEURAL",
    "fullForm": "Intrapleural Injection Full Text",
    "functionalGroup": "Specialized Organs / Cavities",
    "umlsCUI": "C38254",
    "synonyms": [
      "intrapleural full",
      "lung sac cavity"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "INTRAPERICARDIAL",
    "fullForm": "Intrapericardial (Into the heart sac)",
    "functionalGroup": "Specialized Organs / Cavities",
    "umlsCUI": "C38253",
    "synonyms": [
      "intrapericardial",
      "pericardial sac",
      "heart lining injection"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "INTRAVESICAL",
    "fullForm": "Intravesical (Into the urinary bladder)",
    "functionalGroup": "Specialized Organs / Cavities",
    "umlsCUI": "C38256",
    "synonyms": [
      "intravesical",
      "bladder wash",
      "bladder instillation",
      "foley route"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "INTRACORPOREAL",
    "fullForm": "Intracorporeal (Inside a body cavity)",
    "functionalGroup": "Specialized Organs / Cavities",
    "umlsCUI": "C1167232",
    "synonyms": [
      "intracorporeal",
      "inside body cavity",
      "surgical space delivery"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "INTRAVITREAL",
    "fullForm": "Intravitreal (Into the eyeball chamber)",
    "functionalGroup": "Specialized Organs / Cavities",
    "umlsCUI": "C38257",
    "synonyms": [
      "intravitreal",
      "eye injection",
      "vitreous humor injection"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "INTRA-GESTATIONAL SAC",
    "fullForm": "Intra-gestational Sac (Amniotic cavity)",
    "functionalGroup": "Specialized Organs / Cavities",
    "umlsCUI": "C1167171",
    "synonyms": [
      "intra-gestational sac",
      "gestational sac",
      "amniotic fluid space"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "FETAL INJECTION",
    "fullForm": "Intrafetal Injection",
    "functionalGroup": "Specialized Organs / Cavities",
    "umlsCUI": "C1167167",
    "synonyms": [
      "fetal injection",
      "intrafetal",
      "direct fetal therapy"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "INTRALESIONAL",
    "fullForm": "Intralesional (Directly into a localized lesion)",
    "functionalGroup": "Topical / Localized",
    "umlsCUI": "C38221",
    "synonyms": [
      "intralesional",
      "lesion injection",
      "wound injection",
      "cyst injection"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "IL",
    "fullForm": "Intralesional Abbreviation Entry",
    "functionalGroup": "Topical / Localized",
    "umlsCUI": "C38221",
    "synonyms": [
      "il",
      "lesion delivery"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "TRANSDERMAL",
    "fullForm": "Transdermal (Absorbed through skin)",
    "functionalGroup": "Topical / Localized",
    "umlsCUI": "C38306",
    "synonyms": [
      "transdermal",
      "absorbed through skin",
      "skin patch absorption",
      "td"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "TD",
    "fullForm": "Transdermal Short Form",
    "functionalGroup": "Topical / Localized",
    "umlsCUI": "C38306",
    "synonyms": [
      "td",
      "patch application"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "TP",
    "fullForm": "Transdermal Patch / Topical Paste",
    "functionalGroup": "Topical / Localized",
    "umlsCUI": "C0699718",
    "synonyms": [
      "tp",
      "topical patch",
      "skin paste",
      "ointment route"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "AXILLARY",
    "fullForm": "Axillary (Applied via armpit)",
    "functionalGroup": "Topical / Localized",
    "umlsCUI": "C1167215",
    "synonyms": [
      "axillary",
      "armpit application",
      "underarm topical"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "PR",
    "fullForm": "Per Rectum (Rectal)",
    "functionalGroup": "Topical / Localized",
    "umlsCUI": "C38295",
    "synonyms": [
      "per rectum",
      "pr",
      "rectal",
      "suppository",
      "enema"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "VG",
    "fullForm": "Vaginal Delivery",
    "functionalGroup": "Topical / Localized",
    "umlsCUI": "C38311",
    "synonyms": [
      "vg",
      "vaginal",
      "per vaginam",
      "pv"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "PV",
    "fullForm": "Per Vaginam Abbreviation",
    "functionalGroup": "Topical / Localized",
    "umlsCUI": "C38311",
    "synonyms": [
      "pv",
      "vaginal suppository"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "IRR",
    "fullForm": "Irrigation (Wound/cavity wash)",
    "functionalGroup": "Topical / Localized",
    "umlsCUI": "C38255",
    "synonyms": [
      "irrigation",
      "irr",
      "wound wash",
      "cavity rinse"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "IR",
    "fullForm": "Irrigation Code Variant",
    "functionalGroup": "Topical / Localized",
    "umlsCUI": "C38255",
    "synonyms": [
      "ir",
      "rinse cleaning"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "DWELL",
    "fullForm": "Line Dwell / Instillation",
    "functionalGroup": "Topical / Localized",
    "umlsCUI": "C3539828",
    "synonyms": [
      "dwell",
      "line dwell",
      "catheter instillation"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "AERO",
    "fullForm": "Inhalation / Aerosol",
    "functionalGroup": "Respiratory",
    "umlsCUI": "C38245",
    "synonyms": [
      "aero",
      "aerosol",
      "inhalation",
      "breathed in",
      "ih"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "INHALATION",
    "fullForm": "Inhalation Delivery Full Text",
    "functionalGroup": "Respiratory",
    "umlsCUI": "C38245",
    "synonyms": [
      "inhalation text",
      "respiratory delivery"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "IH",
    "fullForm": "Inhalation Initial Abbreviation",
    "functionalGroup": "Respiratory",
    "umlsCUI": "C38245",
    "synonyms": [
      "ih",
      "inhaled formulation"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "NEB",
    "fullForm": "Nebuliser",
    "functionalGroup": "Respiratory",
    "umlsCUI": "C0559648",
    "synonyms": [
      "neb",
      "nebulizer",
      "neb therapy",
      "breathing treatment aerosol"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "ET",
    "fullForm": "Endotracheal Tube Administration",
    "functionalGroup": "Respiratory",
    "umlsCUI": "C1167431",
    "synonyms": [
      "et tube",
      "endotracheal tube",
      "breathing tube delivery"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "INTRANASAL",
    "fullForm": "Intranasal / Nasal Spray",
    "functionalGroup": "Respiratory",
    "umlsCUI": "C38259",
    "synonyms": [
      "intranasal",
      "nasal spray",
      "in nose",
      "nas",
      "in"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "IN",
    "fullForm": "Intranasal Abbreviation Entry",
    "functionalGroup": "Respiratory",
    "umlsCUI": "C38259",
    "synonyms": [
      "in",
      "nasal mist"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "NAS",
    "fullForm": "Nasal Variant Key",
    "functionalGroup": "Respiratory",
    "umlsCUI": "C38259",
    "synonyms": [
      "nas",
      "nasal drop"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "INTRATRACHEAL",
    "fullForm": "Intratracheal Instillation",
    "functionalGroup": "Respiratory",
    "umlsCUI": "C38260",
    "synonyms": [
      "intratracheal",
      "direct airway injection"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "LEFT EAR",
    "fullForm": "Left Ear (Auris Sinistra)",
    "functionalGroup": "Sensory Organs",
    "umlsCUI": "C38202",
    "synonyms": [
      "left ear",
      "as",
      "auris sinistra",
      "ear drops left"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "AS",
    "fullForm": "Auris Sinistra Standard Abbreviation",
    "functionalGroup": "Sensory Organs",
    "umlsCUI": "C38202",
    "synonyms": [
      "as",
      "otic left"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "RIGHT EAR",
    "fullForm": "Right Ear (Auris Dextra)",
    "functionalGroup": "Sensory Organs",
    "umlsCUI": "C38203",
    "synonyms": [
      "right ear",
      "ad",
      "auris dextra",
      "ear drops right"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "AD",
    "fullForm": "Auris Dextra Standard Abbreviation",
    "functionalGroup": "Sensory Organs",
    "umlsCUI": "C38203",
    "synonyms": [
      "ad",
      "otic right"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "BOTH EARS",
    "fullForm": "Both Ears (Aures Uterque)",
    "functionalGroup": "Sensory Organs",
    "umlsCUI": "C38204",
    "synonyms": [
      "both ears",
      "au",
      "aures uterque"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "AU",
    "fullForm": "Aures Uterque Standard Abbreviation",
    "functionalGroup": "Sensory Organs",
    "umlsCUI": "C38204",
    "synonyms": [
      "au",
      "otic both"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "LEFT EYE",
    "fullForm": "Left Eye (Oculus Sinister)",
    "functionalGroup": "Sensory Organs",
    "umlsCUI": "C38205",
    "synonyms": [
      "left eye",
      "os",
      "oculus sinister",
      "eye drops left"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "OS",
    "fullForm": "Oculus Sinister Standard Abbreviation",
    "functionalGroup": "Sensory Organs",
    "umlsCUI": "C38205",
    "synonyms": [
      "os",
      "ophthalmic left"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "RIGHT EYE",
    "fullForm": "Right Eye (Oculus Dexter)",
    "functionalGroup": "Sensory Organs",
    "umlsCUI": "C38206",
    "synonyms": [
      "right eye",
      "od",
      "oculus dexter",
      "eye drops right"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "OD",
    "fullForm": "Oculus Dexter Standard Abbreviation",
    "functionalGroup": "Sensory Organs",
    "umlsCUI": "C38206",
    "synonyms": [
      "od",
      "ophthalmic right"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "BOTH EYES",
    "fullForm": "Both Eyes (Oculi Uterque)",
    "functionalGroup": "Sensory Organs",
    "umlsCUI": "C38207",
    "synonyms": [
      "both eyes",
      "ou",
      "oculi uterque"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "OU",
    "fullForm": "Oculi Uterque Standard Abbreviation",
    "functionalGroup": "Sensory Organs",
    "umlsCUI": "C38207",
    "synonyms": [
      "ou",
      "ophthalmic both"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "INFUSION VIA ON-Q PUMP",
    "fullForm": "On-Q Post-Op Pain Relief Pump System",
    "functionalGroup": "Specialized Equipment",
    "umlsCUI": "C1167385",
    "synonyms": [
      "on-q pump",
      "on q pump",
      "surgical site pain pump infusion"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "IMPELLA",
    "fullForm": "Impella Cardiac Support Device Route",
    "functionalGroup": "Specialized Equipment",
    "umlsCUI": "C4321481",
    "synonyms": [
      "impella circuit",
      "impella infusion",
      "cardiac pump assist route"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "REPLACE",
    "fullForm": "Replacement Fluid Line",
    "functionalGroup": "Specialized Equipment",
    "umlsCUI": "C2348574",
    "synonyms": [
      "replacement fluid",
      "crrt replacement ports",
      "hemofiltration circuit fluid"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "DIALYS",
    "fullForm": "Dialysis Fluid / Intradialytic Route",
    "functionalGroup": "Specialized Equipment",
    "umlsCUI": "C0011786",
    "synonyms": [
      "dialys",
      "dialysis line injection",
      "dialysate interface"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "LOCK",
    "fullForm": "Line Lock (Heparin/Saline cap lock)",
    "functionalGroup": "Specialized Equipment",
    "umlsCUI": "C3539828",
    "synonyms": [
      "line lock",
      "heparin lock",
      "antibiotic catheter lock"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "IMPLANT",
    "fullForm": "Surgical Implant",
    "functionalGroup": "Specialized Equipment",
    "umlsCUI": "C38246",
    "synonyms": [
      "implant",
      "pellet insert",
      "surgical wafer delivery"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "EX-VIVO",
    "fullForm": "Outside the living body (Extracorporeal circuit processing)",
    "functionalGroup": "Specialized Equipment",
    "umlsCUI": "C0392900",
    "synonyms": [
      "ex-vivo",
      "ex vivo",
      "extracorporeal blood cell treatment",
      "outside body circuit"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "AC",
    "fullForm": "Ante Cibum (Before meals) or Intra-coronary contextual tag",
    "functionalGroup": "Miscellaneous / Context Dependent",
    "umlsCUI": "C1554181",
    "synonyms": [
      "ac",
      "before meals",
      "intra-coronary mapping device"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "NU",
    "fullForm": "Nasouretic Tube / Nutritional Route variation",
    "functionalGroup": "Miscellaneous / Context Dependent",
    "umlsCUI": "C1167439",
    "synonyms": [
      "nu",
      "nutrition line index"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "NS",
    "fullForm": "Normal Saline Route variant / Nasal Spray tag",
    "functionalGroup": "Miscellaneous / Context Dependent",
    "umlsCUI": "C38259",
    "synonyms": [
      "ns",
      "normal saline flush route",
      "nasal spray base"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "TT",
    "fullForm": "Tracheostomy Tube or Trans-Tracheal routing",
    "functionalGroup": "Miscellaneous / Context Dependent",
    "umlsCUI": "C1167431",
    "synonyms": [
      "tt",
      "trach route",
      "tracheostomy tube delivery"
    ]
  },
  {
    "@type": "RouteVocabulary",
    "rawCode": "null",
    "fullForm": "Unknown / Missing Value",
    "functionalGroup": "Data Status",
    "umlsCUI": "C1299581",
    "synonyms": [
      "null",
      "unknown",
      "missing",
      "not recorded",
      "none",
      "blank"
    ]
  }
]
"""

records = json.loads(ROUTE_VOCABULARY_JSON)
expected_rows = len(records)

print(f"Embedded medication-route records: {expected_rows}")
print(f"Target table: {TARGET_TABLE}")

## 1. Validate the target schema

The notebook expects the Gold schema to already exist.
It does not create or modify any other schema or table.

In [0]:
schema_exists = (
    spark.sql("SHOW SCHEMAS IN cdr_mimiciv")
         .filter(F.col("databaseName") == "gold")
         .count()
    > 0
)

if not schema_exists:
    raise RuntimeError(
        "Schema cdr_mimiciv.gold does not exist. "
        "Create/confirm the Gold schema before running this notebook."
    )

print("Validated: cdr_mimiciv.gold exists.")

## 2. Build the reference DataFrame

In [0]:
source_schema = T.StructType([
    T.StructField("@type", T.StringType(), False),
    T.StructField("rawCode", T.StringType(), False),
    T.StructField("fullForm", T.StringType(), False),
    T.StructField("functionalGroup", T.StringType(), False),
    T.StructField("umlsCUI", T.StringType(), False),
    T.StructField("synonyms", T.ArrayType(T.StringType()), False),
])

source_df = spark.createDataFrame(records, schema=source_schema)

route_df = (
    source_df
    .select(
        F.col("rawCode").alias("route_code"),
        F.upper(F.trim(F.col("rawCode"))).alias("route_code_normalized"),
        F.col("fullForm").alias("route_name"),
        F.col("functionalGroup").alias("functional_group"),
        F.col("umlsCUI").alias("umls_cui"),
        F.col("synonyms"),
        F.concat_ws(", ", F.col("synonyms")).alias("synonyms_text"),
    )
    .withColumn(
        "semantic_search_text",
        F.concat_ws(
            " | ",
            F.col("route_code"),
            F.col("route_name"),
            F.col("functional_group"),
            F.col("synonyms_text"),
        ),
    )
    .withColumn(
        "source_vocabulary",
        F.lit("MIMIC-IV CLIF medication route semantic reference"),
    )
)

display(route_df.orderBy("functional_group", "route_code"))

## 3. Data-quality checks

Before writing to Gold, validate that:
- every route has a code,
- every route code is unique,
- required descriptive fields are present,
- the transformed row count matches the embedded source vocabulary.

Note: the source includes the literal route code `"null"` to represent an unknown/missing route category.
It is intentionally preserved as a vocabulary value rather than converted to SQL `NULL`.

In [0]:
actual_rows = route_df.count()
null_code_rows = route_df.filter(F.col("route_code").isNull()).count()
duplicate_code_rows = (
    route_df.groupBy("route_code")
            .count()
            .filter(F.col("count") > 1)
            .count()
)
missing_required_rows = route_df.filter(
    F.col("route_name").isNull()
    | F.col("functional_group").isNull()
    | F.col("umls_cui").isNull()
    | F.col("synonyms").isNull()
).count()

assert actual_rows == expected_rows, (
    f"Row-count mismatch: expected {expected_rows}, got {actual_rows}"
)
assert null_code_rows == 0, f"Found {null_code_rows} SQL NULL route codes"
assert duplicate_code_rows == 0, f"Found {duplicate_code_rows} duplicate route codes"
assert missing_required_rows == 0, (
    f"Found {missing_required_rows} rows missing required descriptive fields"
)

print("Validation passed.")
print(f"Rows: {actual_rows}")
print(f"SQL NULL route codes: {null_code_rows}")
print(f"Duplicate route codes: {duplicate_code_rows}")
print(f"Rows missing required fields: {missing_required_rows}")

## 4. Create or replace the Gold dimension

This operation only replaces the target table:

`cdr_mimiciv.gold.dim_medication_route`

No Bronze, Silver, or other Gold table is modified.

In [0]:
route_df.createOrReplaceTempView("vw_dim_medication_route_build")

spark.sql(f"""
CREATE OR REPLACE TABLE {TARGET_TABLE}
USING DELTA
COMMENT 'Gold semantic reference dimension for MIMIC-IV medication administration route codes, route names, functional groups, UMLS CUIs, and synonyms. Intended for governed analytics, AI/BI, and Genie.'
AS
SELECT
    route_code,
    route_code_normalized,
    route_name,
    functional_group,
    umls_cui,
    synonyms,
    synonyms_text,
    semantic_search_text,
    source_vocabulary
FROM vw_dim_medication_route_build
""")

print(f"Created/replaced {TARGET_TABLE}")

## 5. Add column documentation

These comments improve discoverability in Catalog Explorer and provide semantic context for AI/BI and Genie.

In [0]:
column_comments = {
    "route_code": "Raw medication administration route code represented by this reference record.",
    "route_code_normalized": "Upper-cased and trimmed route code for consistent analytical joins.",
    "route_name": "Human-readable medication administration route name.",
    "functional_group": "Higher-level route category such as Enteral / Oral, Intravenous (IV), Respiratory, or Topical / Localized.",
    "umls_cui": "UMLS Concept Unique Identifier supplied by the reference vocabulary.",
    "synonyms": "Array of alternative clinical or natural-language terms associated with the route.",
    "synonyms_text": "Comma-separated synonym text for easier BI display and semantic interpretation.",
    "semantic_search_text": "Combined route code, name, group, and synonyms intended to support natural-language interpretation.",
    "source_vocabulary": "Provenance label for the embedded medication-route semantic reference vocabulary.",
}

for column_name, comment in column_comments.items():
    safe_comment = comment.replace("'", "''")
    spark.sql(
        f"ALTER TABLE {TARGET_TABLE} "
        f"ALTER COLUMN {column_name} COMMENT '{safe_comment}'"
    )

print("Column comments applied.")

## 6. Post-write validation and preview

In [0]:
written_df = spark.table(TARGET_TABLE)
written_rows = written_df.count()

assert written_rows == expected_rows, (
    f"Post-write row-count mismatch: expected {expected_rows}, got {written_rows}"
)

print(f"Post-write validation passed: {written_rows} rows")
display(
    written_df
    .select(
        "route_code",
        "route_name",
        "functional_group",
        "umls_cui",
        "synonyms_text",
    )
    .orderBy("functional_group", "route_code")
)

## 7. Example analytical use

The dimension can be joined to a medication fact or prescription dataset using a normalized route value.

Example pattern:

```sql
SELECT
  m.*,
  r.route_name,
  r.functional_group,
  r.umls_cui
FROM cdr_mimiciv.gold.<medication_fact> AS m
LEFT JOIN cdr_mimiciv.gold.dim_medication_route AS r
  ON UPPER(TRIM(m.route)) = r.route_code_normalized;
```

Replace `<medication_fact>` with the actual Gold medication table after confirming its route column.